In [0]:
from pyspark.sql.functions import *

loading_folder_name = '/Volumes/ws_azuredatabricks/raw/landing/'
bronze_schema_name = 'ws_azuredatabricks.bronze'

source_file_names = ["patient_raw", "visit_raw", "diagnosis_raw", "hospital_raw"]

for file_name in source_file_names:
    bronze_table_name = file_name.lower().replace('_raw','')

    raw_dataframe =  (
        spark
        .read
        .options(header='true', inferSchema='true')
        .csv(f'{loading_folder_name}{file_name}.csv')
    )

    clean_column_name = [
        col_name.strip().lower().replace(' ', '_') for col_name in raw_dataframe.columns
    ]

    raw_dataframe = raw_dataframe.toDF(*clean_column_name)

    bronze_dataframe = raw_dataframe.write.mode('overwrite').saveAsTable(f'{bronze_schema_name}.{bronze_table_name}')

    print(f'Created {bronze_schema_name}.{bronze_table_name}')
    

Created ws_azuredatabricks.bronze.patient
Created ws_azuredatabricks.bronze.visit
Created ws_azuredatabricks.bronze.diagnosis
Created ws_azuredatabricks.bronze.hospital


In [0]:
for tab in spark.catalog.listTables(bronze_schema_name):
    print(f'{bronze_schema_name}.{tab.name}')
    display(spark.table(f'{bronze_schema_name}.{tab.name}'))

ws_azuredatabricks.bronze.diagnosis


diagnosis_code,diagnosis_desc
D001,Hypertension
D002,Diabetes Type 2
D003,Chest Pain
D004,Asthma
D005,Kidney Infection


ws_azuredatabricks.bronze.hospital


hospital_id,hospital_name,city,bed_count
H001,Apollo Main Hospital,Chennai,850
H002,Fortis Healthcare Delhi,Delhi,600
H003,Kokilaben Dhirubhai Hospital,Mumbai,700
H004,Manipal Hospital,Bangalore,400
H005,Yashoda Hospital,Hyderabad,450


ws_azuredatabricks.bronze.patient


patient_id,first_name,last_name,gender,dob,city
P001,Aarav,Sharma,M,1980-02-14,Mumbai
P002,Riya,Verma,F,1975-07-22,Delhi
P003,Kabir,Menon,M,1990-11-02,Bangalore
P004,Sneha,Rao,F,1988-09-19,Hyderabad
P005,Vikram,Singh,M,1965-05-05,Chennai


ws_azuredatabricks.bronze.visit


visit_id,patient_id,hospital_id,admission_date,discharge_date,diagnosis_code,cost
V1001,P001,H003,2025-01-01,2025-01-05,D003,12000
V1002,P001,H003,2025-01-20,2025-01-22,D003,9000
V1003,P002,H002,2025-02-10,2025-02-15,D001,18000
V1004,P002,H002,2025-04-01,2025-04-05,D001,17000
V1005,P003,H004,2025-03-12,2025-03-18,D005,22000
V1006,P003,H004,2025-03-28,2025-03-31,D005,16000
V1007,P004,H005,2025-01-15,2025-01-20,D004,14000
V1008,P005,H001,2025-02-25,2025-03-02,D002,20000
